## About
- This notebook includes training and evaluation of all two-feature combinations using symbolic regression

### Input
1. curated proteomics and clinical data
2. shortlisted and novel cytokines

### Output
> All single feature and two-feature combination models
>
> performance metrics across training, test and validation sets of the above models

### Model training and selection structure

```
Full Dataset
|── Training Cohort (combined)
|    ├── Training Set (80%)
|    └── Test Set (20%)   ← used for model selection
└── Independent Validation Cohort
```


In [ ]:
# Load packages 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import logging
import feyn
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve
import itertools
from tqdm import tqdm
from datetime import datetime
import ast
logger = logging.getLogger()
logger.setLevel(logging.INFO)
import pickle

### Import curated proteomics and clinical data

In [ ]:
# Define project directory
Base = 'data_directory'

df_olink = pd.read_excel(os.path.join(Base, 'data_curated/cureated_proteomics_clinical_data.xlsx'), sheet_name='proteomics_curated').set_index('CBMRID')
df_cli = pd.read_excel(os.path.join(Base, 'data_curated/cureated_proteomics_clinical_data.xlsx'), sheet_name='clinical_curated').set_index('CBMRID')

In [ ]:
# Join curated proteomics and clinical data
data = df_olink.join(df_cli)
# Drop samples without outcome target measure
data = data[data['cohort']!='GALA_LiHep'].dropna(subset=['inflam_binary'])

In [ ]:
olink_proteins = df_olink.columns
addon = ['alt_log2', 'ast_log2']

In [ ]:
# import MS data 
data_ms = pd.read_csv(os.path.join(Base, 'data_curated/olink_ms_combined.csv')).set_index('CBMRID')
data=data.join(data_ms[['Q08380']])

In [ ]:
olink_ms_proteins = olink_proteins.tolist() + ['Q08380']
olink_ms_proteins_altast = olink_ms_proteins+addon

### Cohorts

In [ ]:
rdc_ids=data[data['cohort']=='RDC'].index
ald_ids=data[data['cohort']=='ALD'].index
hp_ids=data[data['cohort']=='HP'].index
sip_ids=data[data['cohort']=='SIP'].index
lihep_ids = data[data['cohort']=='GALA_LiHep'].index
sip_biop_ids = data.iloc[data.index.isin(sip_ids)][['lobinflammation', 'ballooning']].dropna().index
ikkesupernomral_ids = data[data['supernormal_filter']].index

### Targets

In [ ]:
# rename binary columns
data['I2'] = data['inflam_binary']

In [ ]:
TARGETS = ['I2']
Y = data[TARGETS]
Y['I2'].value_counts(dropna=False)

### Shortlisted and novel cytokines 

In [ ]:
shortlist = pd.read_csv(os.path.join(Base, 'mrmr/shortlisted_biomarkers_by_mrmr.csv'))
shortlisted=shortlist['Protein'].tolist()
novel = shortlist[shortlist['novel']]['Protein']

In [ ]:
from sklearn.metrics import confusion_matrix, roc_auc_score, f1_score, precision_score, recall_score, accuracy_score, balanced_accuracy_score, matthews_corrcoef

def get_scoring(y_true, y_score, y_pred):
    scores = []
    cm = confusion_matrix(y_true, y_pred)
    tn = cm[0][0]
    fn = cm[1][0]
    npv = tn/(tn+fn)
    roc_auc = roc_auc_score(y_true, y_score)
    f1 = f1_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    accuracy = accuracy_score(y_true, y_pred)
    balanced_accuracy = balanced_accuracy_score(y_true, y_pred)
    specificity = 2 * balanced_accuracy - recall
    mcc = matthews_corrcoef(y_true, y_pred) 
    scores = [roc_auc, f1, precision, npv, recall, accuracy, balanced_accuracy, specificity, mcc]
    names = ['roc_auc', 'f1', 'precision','NPV', 'recall', 'accuracy', 'balanced accuracy', 'specificity', 'mcc']
    scores = pd.DataFrame(dict(zip(names, scores)), index=[0])
    return scores.round(3)

### QLattice

### All two-marker combination models

In [ ]:
target='I2'
dataset=data[olink_proteins.tolist() + addon + [target]].dropna()

In [ ]:
_train = dataset[dataset.index.isin(ald_ids.tolist() + hp_ids.tolist() + rdc_ids.tolist())]
train, test = train_test_split(_train, random_state=42, test_size=.2, stratify=_train['I2'])
validation1 = dataset[dataset.index.isin(sip_ids)].dropna()
validation2 = dataset[dataset.index.isin(sip_biop_ids)].dropna()

test_sn = test.drop(set(ikkesupernomral_ids) & set(test.index))
validation1_sn = validation1.drop(set(ikkesupernomral_ids) & set(validation1.index))
validation2_sn = validation2.drop(set(ikkesupernomral_ids) & set(validation2.index))

In [ ]:
print('shape of training set: {}'.format(train.shape))
print('shape of test set: {}'.format(test.shape))
print('shape of test supernormal set: {}'.format(test_sn.shape))
print('shape of validation1 set: {}'.format(validation1.shape))
print('shape of validation1 supernormal set: {}'.format(validation1_sn.shape))
print('shape of validation2 set: {}'.format(validation2.shape))
print('shape of validation2 supernormal set: {}'.format(validation2_sn.shape))

In [ ]:
cohort_index_dict = {'train':train.index, 'test':test.index, 'validation1':validation1.index, 'validation2':validation2.index}
with open(os.path.join(Base, 'symbolic_regression/splits.pkl'), 'wb') as f:
    pickle.dump(cohort_index_dict, f)

In [ ]:
shortlisted2 = list(shortlisted)+addon
#shortlisted2 = [i for i in shortlisted2 if i!='Q08380']

In [ ]:
combos = []
for i in itertools.combinations(shortlisted2, 2):
    combos.append(i)

In [ ]:
file_combo_models = os.path.join(Base, 'symbolic_regression/models/combo_models_v2.pkl')

RE_COMPUTE = False
if not RE_COMPUTE:
    with open(file_combo_models, 'rb') as f:
        combo_models = pickle.load(f)
else:    
    combo_models = {}
    n=1
    combo_logfile=os.path.join(Base, 'symbolic_regression/log/run_combo.log')
    with open(combo_logfile, 'w') as file:
        now=datetime.now()
        current_time=now.strftime("%D %H:%M:%S")
        file.write("Analysis started at {}\n".format(current_time))
        file.flush()
        for combo in tqdm(combos):
            selected_features = list(combo)
            file.write("combo{}: {}\n".format(n, selected_features))
            file.flush()
            print("combo{}: {}".format(n, selected_features))
            n+=1
            train_combo = train[selected_features + ['I2']]
            ql = feyn.QLattice(random_seed=42)
            models = ql.auto_run(
                data=train_combo,
                output_name='I2',
                query_string = "_{}".format(selected_features),
                kind='classification'
            )
            #best=models[0]
            combo_models[combo]=models[0]
        now=datetime.now()
        current_time=now.strftime("%D %H:%M:%S")
        file.write('Analysis done at {}'.format(current_time))
        file.flush()
    with open(file_combo_models, 'wb') as f:
        pickle.dump(combo_models, f)

In [ ]:
model_performance_combo= []
roc_curves_combo_i = {}
for combo in combo_models.keys():
    model = combo_models[combo]
    formula = str(model.sympify(symbolic_lr=True, include_weights=True))
    features = model.features
    datasets = {"train":train, "test":test, 
                "test_sn":test_sn,
                'validation1':validation1, 
                "validation1_sn":validation1_sn,
                'validation2':validation2,
                "validation2_sn":validation2_sn
               }
    roc_curves = {}
    for split in datasets.keys():
        y_true = datasets[split][target]
        y_score = model.predict(datasets[split])
        y_pred = y_score.round()
        roc_curves[split]=(y_true, y_score, y_pred)
        nr_obs=len(y_pred)
        class1_pert = y_true.sum()/nr_obs*100
        scoring = get_scoring(y_true, y_score, y_pred)
        scoring['nr_obs']=nr_obs
        scoring['class1_pert']=class1_pert.astype(int)
        scoring['set']=split
        scoring['features']=str(features)
        scoring['formula']=formula
        scoring['nr_features']=len(features)
        scoring['feature1']=ast.literal_eval(str(features))[0]
        scoring['feature2']=ast.literal_eval(str(features))[1]
        contains_novel_cytokines = list(set(features) & set(novel))
        if len(contains_novel_cytokines)>0:
            contains_novel = True
        else:
            contains_novel = False
        scoring['novel cytokines']=str(contains_novel_cytokines)
        scoring['nr_novel_features']=len(contains_novel_cytokines)
        scoring['contains novel']=contains_novel
        scoring['model']=str(model)
        model_performance_combo.append(scoring)
    roc_curves_combo_i[combo]=roc_curves
model_performance_combo = pd.concat(model_performance_combo)

file_combo_roc_aucs = os.path.join(Base, 'symbolic_regression/models/roc_curves_combo_v3.pkl')
with open(file_combo_roc_aucs , 'wb') as f:
    pickle.dump(roc_curves_combo_i, f)

#### All combinations containing LGALS3BP

In [ ]:
combos_ms = [('Q08380', i) for i in shortlisted2 if i!='Q08380']

In [ ]:
dataset_ms = dataset.join(data_ms[['Q08380']]).dropna()
train_ms, test_ms = train_test_split(dataset_ms, random_state=42, test_size=0.2, stratify=dataset_ms['I2'])

In [ ]:
file_combo_models_ms = os.path.join(Base, 'symbolic_regression/models/combo_models_ms_v2.pkl')

RE_COMPUTE = False
if not RE_COMPUTE:
    with open(file_combo_models_ms, 'rb') as f:
        combo_models_ms = pickle.load(f)
else:
    combo_ms_logfile=os.path.join(Base, 'symbolic_regression/log/run_combo_ms.log')
    combo_models_ms = {}
    n=1
    with open(combo_ms_logfile, 'w') as file:
        now=datetime.now()
        current_time=now.strftime("%D %H:%M:%S")
        file.write("Analysis started at {}\n".format(current_time))
        file.flush()
        for combo in tqdm(combos_ms):
            selected_features = list(combo)
            file.write("combo{}: {}\n".format(n, selected_features))
            file.flush()
            print("combo{}: {}".format(n, selected_features))
            n+=1
            train_combo = train_ms[selected_features + ['I2']]
            ql = feyn.QLattice(random_seed=42)
            models = ql.auto_run(
                data=train_combo,
                output_name='I2',
                query_string = "_{}".format(selected_features),
                kind='classification'
            )
            #best=models[0]
            combo_models_ms[combo]=models[0]
        now=datetime.now()
        current_time=now.strftime("%D %H:%M:%S")
        file.write('Analysis done at {}'.format(current_time))
        file.flush()
    with open(file_combo_models_ms, 'wb') as f:
        pickle.dump(combo_models_ms, f)

In [ ]:
model_performance_combo_ms= []
roc_curves_combo_ms_i={}

for combo in combo_models_ms.keys():
    model = combo_models_ms[combo]
    features = model.features
    formula = str(model.sympify(symbolic_lr=True, include_weights=True))
    datasets = {"train_ms":train_ms, 'test_ms':test_ms}
    roc_curves = {}
    for split in ["train_ms", "test_ms"]:
        y_true = datasets[split][target]
        y_score = model.predict(datasets[split])
        y_pred = y_score.round()
        roc_curves[split.split('_')[0]]=(y_true, y_score, y_pred)
        nr_obs=len(y_pred)
        class1_pert = y_true.sum()/nr_obs*100
        scoring = get_scoring(y_true, y_score, y_pred)
        scoring['nr_obs']=nr_obs
        scoring['class1_pert']=class1_pert.astype(int)
        scoring['set']=split
        scoring['features']=str(features)
        scoring['formula']=formula
        scoring['nr_features']=len(features)
        scoring['feature1']=ast.literal_eval(str(features))[0]
        scoring['feature2']=ast.literal_eval(str(features))[1]
        contains_novel_cytokines = list(set(features) & set(novel))
        if len(contains_novel_cytokines)>0:
            contains_novel = True
        else:
            contains_novel = False
        scoring['novel cytokines']=str(contains_novel_cytokines)
        scoring['nr_novel_features']=len(contains_novel_cytokines)
        scoring['contains novel']=contains_novel
        scoring['model']=str(model)
        model_performance_combo_ms.append(scoring)    
    roc_curves_combo_ms_i[combo]=roc_curves
model_performance_combo_ms = pd.concat(model_performance_combo_ms)

file_combo_ms_roc_aucs = os.path.join(Base, 'symbolic_regression/models/roc_curves_combo_ms_v3.pkl')
with open(file_combo_ms_roc_aucs , 'wb') as f:
    pickle.dump(roc_curves_combo_ms_i, f)

#### LGALS3BP alone

In [ ]:
file_model_ms = os.path.join(Base, 'symbolic_regression/models/model_ms_v2.pkl')
single_features_ms = ['Q08380', 'alt_log2', 'ast_log2']

RE_COMPUTE = False
if not RE_COMPUTE:
    with open(file_model_ms, 'rb') as f:
        single_feature_models_ms = pickle.load(f)
        
else:
    single_feature_models_ms = {}
    for feature in single_features_ms:
        ql = feyn.QLattice(random_seed=42)
        models = ql.auto_run(
            data=train_ms[[feature, 'I2']],
            output_name='I2',
            query_string = "_{}".format([feature]),
            kind='classification'
        )
        single_feature_models_ms[feature]=models[0]
    with open(file_model_ms, 'wb') as f:
        pickle.dump(single_feature_models_ms, f)

In [ ]:
model_performance_ms = []
datasets = {"train_ms":train_ms, 'test_ms':test_ms}
roc_curves_singlefeature_ms = {}

for feature in single_feature_models_ms.keys():
    model = single_feature_models_ms[feature]
    features = model.features
    formula = str(model.sympify(symbolic_lr=True, include_weights=True))
    roc_curves={}
    for split in ["train_ms", "test_ms"]:
        y_true = datasets[split][target]
        y_score = model.predict(datasets[split])
        y_pred = y_score.round()
        roc_curves[split.split('_')[0]]=(y_true, y_score, y_pred)
        nr_obs=len(y_pred)
        class1_pert = y_true.sum()/nr_obs*100
        scoring = get_scoring(y_true, y_score, y_pred)
        scoring['nr_obs']=nr_obs
        scoring['class1_pert']=class1_pert.astype(int)
        scoring['set']=split
        scoring['features']=features[0] +'_ms'
        scoring['formula']=formula
        scoring['nr_features']=len(features)
        scoring['feature1']=ast.literal_eval(str(features))[0] +'_ms'
        scoring['feature2']=ast.literal_eval(str(features))[0] +'_ms'
        contains_novel_cytokines = list(set(features) & set(novel))
        if len(contains_novel_cytokines)>0:
            contains_novel = True
        else:
            contains_novel = False
        scoring['novel cytokines']=str(contains_novel_cytokines)
        scoring['nr_novel_features']=len(contains_novel_cytokines)
        scoring['contains novel']=contains_novel
        scoring['model']=str(model)
        model_performance_ms.append(scoring)
    roc_curves_singlefeature_ms[feature+'_ms']=roc_curves

In [ ]:
model_ms_alone = pd.concat(model_performance_ms)

#### one feature models

In [ ]:
file_single_feature_models = os.path.join(Base, 'symbolic_regression/models/single_feature_models_v2.pkl')

RE_COMPUTE = False
if not RE_COMPUTE:
    with open(file_single_feature_models, 'rb') as f:
        single_feature_models = pickle.load(f)

else:
    single_feature_models = {}
    n=1
    onefeature_logfile=os.path.join(Base, 'symbolic_regression/log/run_single_feature.log')
    with open(onefeature_logfile, 'w') as file:
        now=datetime.now()
        current_time=now.strftime("%D %H:%M:%S")
        file.write("Analysis started at {}\n".format(current_time))
        for feature in tqdm(shortlisted2):      
            file.write("feature{}: {}\n".format(n, feature))
            file.flush()
            print("feature{}: {}\n".format(n, feature))
            n+=1
            ql = feyn.QLattice(random_seed=42)
            models = ql.auto_run(
                data=train[[feature, 'I2']],
                output_name='I2',
                query_string = "_{}".format([feature]),
                kind='classification'
            )  
            single_feature_models[feature]=models[0]
            now=datetime.now()
        current_time=now.strftime("%D %H:%M:%S")
        file.write('Analysis done at {}'.format(current_time))
        file.flush()
    with open(file_single_feature_models, 'wb') as f:
        pickle.dump(single_feature_models, f)

#### Model performance

In [ ]:
model_performance_singlefeature= []
roc_curves_singlefeature_i = {}
for feature in single_feature_models.keys():
    model = single_feature_models[feature]
    features = model.features
    formula = str(model.sympify(symbolic_lr=True, include_weights=True))
    datasets = {"train":train, "test":test, 
                "test_sn":test_sn,
                'validation1':validation1, 
                "validation1_sn":validation1_sn,
                'validation2':validation2,
                "validation2_sn":validation2_sn
               }
    roc_curves={}
    for split in datasets.keys():
        y_true = datasets[split][target]
        y_score = model.predict(datasets[split])
        y_pred = y_score.round()
        roc_curves[split]=(y_true, y_score, y_pred)
        nr_obs=len(y_pred)
        class1_pert = y_true.sum()/nr_obs*100
        scoring = get_scoring(y_true, y_score, y_pred)
        scoring['nr_obs']=nr_obs
        scoring['class1_pert']=class1_pert.astype(int)
        scoring['set']=split
        scoring['features']=str(features)
        scoring['formula']=formula
        scoring['feature1']=ast.literal_eval(str(features))[0]
        scoring['feature2']=ast.literal_eval(str(features))[0]
        scoring['nr_features']=len(features)
        contains_novel_cytokines = list(set(features) & set(novel))
        if len(contains_novel_cytokines)>0:
            contains_novel = True
        else:
            contains_novel = False
        scoring['novel cytokines']=str(contains_novel_cytokines)
        scoring['nr_novel_features']=len(contains_novel_cytokines)
        scoring['contains novel']=contains_novel
        scoring['model']=str(model)
        model_performance_singlefeature.append(scoring)   
    roc_curves_singlefeature_i[feature]=roc_curves
model_performance_singlefeature = pd.concat(model_performance_singlefeature)

file_singlefeature_roc_curves = os.path.join(Base, 'symbolic_regression/models/roc_curves_singlefeature_v3.pkl')
with open(file_singlefeature_roc_curves , 'wb') as f:
    pickle.dump(roc_curves_singlefeature_i, f)

#### Bootstrap to calculate confidence intervals at 2.5th and 97.5th

In [ ]:
top7 = [('HGF', 'SCF'), 
       ('HGF', 'TWEAK'),
       ('HGF', 'TRAIL'),
       ('SCF', 'SLAMF1'),
       ('HGF', 'ast_log2'),
       'ast_log2',
       'alt_log2']

In [ ]:
from sklearn.utils import resample
n_boot = 1000
boot_all = []

for combo in tqdm(top7):
    boot_res = []
    if type(combo) is tuple:
        model = combo_models[combo]
    else:
        model = single_feature_models[combo]
        
    for i in range(n_boot):
    
        # Bootstrap validation cohort
        boot_df = resample(
            validation1,
            replace=True,
            n_samples=len(validation1),
            random_state=i,
            stratify=validation1[target]
        )
    
        # Predictions
        y_true = boot_df[target]
        y_score = model.predict(boot_df)
        y_pred = y_score.round()
    
        # Compute all metrics
        scoring = get_scoring(y_true, y_score, y_pred)
        boot_res.append(scoring)
    
    df_res = pd.concat(boot_res)
    
    ci = pd.DataFrame({
        "estimate": df_res.mean(),
        "lower95": df_res.quantile(0.025),
        "upper95": df_res.quantile(0.975)
    })
    ci_model = ci.T.copy()
    ci_model['features']=str(combo)
    boot_all.append(ci_model)

In [ ]:
df_boot = pd.concat(boot_all)
df_boot.to_csv(os.path.join(Base, 'symbolic_regression/models/ci_estimate.csv'))

### Combine model performance

In [ ]:
two_feature_combo_olink_ms = pd.concat([model_performance_combo,
                                        #model_performance_combo_ms,
                                        #model_ms_alone,
                                        model_performance_singlefeature])
two_feature_combo_olink_ms.index=np.arange(two_feature_combo_olink_ms.shape[0])

In [ ]:
# Save model performance
two_feature_combo_olink_ms.to_excel(os.path.join(Base, 'symbolic_regression/models/model_performance_all_v3.xlsx'))

### Combine roc curves for all models

In [ ]:
roc_curves_combined = roc_curves_combo_i | roc_curves_combo_ms_i | roc_curves_singlefeature_ms | roc_curves_singlefeature_i

In [ ]:
roc_curves_combined = roc_curves_combo_i | roc_curves_singlefeature_i

In [ ]:
roc_curves_combined_file = os.path.join(Base, 'symbolic_regression/models/roc_curves_combined_v3.pkl')
with open(roc_curves_combined_file , 'wb') as f:
    pickle.dump(roc_curves_combined, f)

In [ ]:
two_feature_combo_olink_ms['nr_obs'].unique()

In [ ]:
two_feature_combo_olink_ms[['set', 'nr_obs']].drop_duplicates()